# ROS 2 Interop with Roboflow Inference Workflows

This notebook demonstrates the rosbridge interop landed in
`feat/rosbridge-interop`. We will:

1. Bring up `rosbridge_server` in Docker.
2. Define a Workflow that consumes a `rosbridge://` image topic, runs
   `rfdetr-medium` object detection, and republishes the detections plus
   an annotated image overlay back onto ROS topics.
3. Submit that Workflow to the inference server's
   `/inference_pipelines/initialise` endpoint so the pipeline runs
   **server-side** (the server is the one that needs the `[ros]` extra).
4. Open RViz 2 to visualize the input camera, the annotated overlay, and
   the `vision_msgs/Detection2DArray` topic.
5. Replay a sample video as if it were a rosbag, by JPEG-encoding frames
   and publishing them as `sensor_msgs/CompressedImage` over rosbridge.

## Architecture

```
  notebook publisher  ──►  rosbridge_server  ──►  inference server
        ▲                    (ws://:9090)         (subscribes to camera,
        │                          ▲               runs workflow,
        │                          │               publishes detections
        └──── RViz 2 subscribes ───┘               + annotated image)
```

Both the inference server and `rosbridge_server` need to share a network
so the server can resolve `localhost:9090`. The notebook uses
`--network host` everywhere.

## 0. Configuration

Set your inference server URL, API key, and the rosbridge endpoint.
If both the inference server and rosbridge run on this host with
`--network host`, the defaults below already work.

In [ ]:
import os

# Inference server with the [ros] extra installed.
API_URL = os.environ.get('API_URL', 'http://localhost:9001')
API_KEY = os.environ.get('ROBOFLOW_API_KEY', 'YOUR_ROBOFLOW_API_KEY')

# Rosbridge — must be reachable from BOTH this notebook and the inference
# server. With --network host on Linux, 'localhost' works for both.
ROSBRIDGE_HOST = os.environ.get('ROSBRIDGE_HOST', 'localhost')
ROSBRIDGE_PORT = int(os.environ.get('ROSBRIDGE_PORT', '9090'))

# Topics
CAMERA_TOPIC = '/camera/image_raw/compressed'  # publisher → server
DETECTIONS_TOPIC = '/inference/detections'       # server → consumer
ANNOTATED_TOPIC = '/inference/image_annotated'   # server → consumer (Compressed)

MODEL_ID = 'rfdetr-medium'  # public Roboflow alias → coco/40

print(f'Inference: {API_URL}')
print(f'Rosbridge: ws://{ROSBRIDGE_HOST}:{ROSBRIDGE_PORT}')

## 1. Install client-side Python deps

We need:

- `roslibpy` — to publish frames into rosbridge from this notebook.
- `opencv-python-headless` — to decode the demo video.
- `requests` — to talk to the inference server's HTTP API.

The inference **server** doesn't need any of these; it has the `[ros]`
extra (which provides `roslibpy` server-side).

In [ ]:
%pip install --quiet roslibpy 'opencv-python-headless>=4.5' requests

## 2. Start `rosbridge_server`

We run rosbridge inside the stock `ros:humble` image on host network.
First run takes ~60 seconds because it `apt install`s rosbridge into
the container. Subsequent runs hit the daemon cache.

If port 9090 is already in use this will fail — kill whatever is
binding it first (`docker rm -f rosbridge` if you have a leftover).

In [ ]:
import subprocess, time

subprocess.run(['docker', 'rm', '-f', 'rosbridge'],
               check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

subprocess.run([
    'docker', 'run', '-d', '--name', 'rosbridge', '--network', 'host',
    'ros:humble',
    'bash', '-c',
    'apt update -qq && apt install -y -qq ros-humble-rosbridge-server && '
    'source /opt/ros/humble/setup.bash && '
    'ros2 launch rosbridge_server rosbridge_websocket_launch.xml'
], check=True)

print('Container starting; waiting for WebSocket on port', ROSBRIDGE_PORT, '…')

In [ ]:
import socket

deadline = time.time() + 180
while time.time() < deadline:
    try:
        with socket.create_connection((ROSBRIDGE_HOST, ROSBRIDGE_PORT), timeout=1):
            print('rosbridge port is open')
            break
    except OSError:
        time.sleep(2)
else:
    raise RuntimeError('rosbridge_server did not come up; check `docker logs rosbridge`')

## 3. Open a `roslibpy` connection

We'll reuse this single connection for both the publisher (Step 6) and
any debugging subscriptions you want to add. The server-side workflow
opens its own connection over the same WebSocket.

In [ ]:
import roslibpy

ros = roslibpy.Ros(host=ROSBRIDGE_HOST, port=ROSBRIDGE_PORT)
ros.run()
assert ros.is_connected, 'roslibpy failed to connect'
print('roslibpy connected:', ros.is_connected)

## 4. Define and submit the Workflow

Three things happen in the Workflow on every frame:

1. `roboflow_object_detection_model@v2` runs `rfdetr-medium`.
2. `bounding_box_visualization@v1` overlays boxes on the input frame.
3. Two `rosbridge_publish_sink@v1` blocks publish:
   - `vision_msgs/Detection2DArray` → `/inference/detections`
   - `sensor_msgs/CompressedImage` → `/inference/image_annotated/compressed`

The pipeline's *video source* is also rosbridge — the inference server
subscribes to `/camera/image_raw/compressed` and feeds frames into the
Workflow's `image` input.

In [ ]:
WORKFLOW = {
    'version': '1.0',
    'inputs': [
        {'type': 'InferenceImage', 'name': 'image'},
    ],
    'steps': [
        {
            'type': 'roboflow_core/roboflow_object_detection_model@v2',
            'name': 'model',
            'images': '$inputs.image',
            'model_id': MODEL_ID,
            'confidence': 0.5,
        },
        {
            'type': 'roboflow_core/bounding_box_visualization@v1',
            'name': 'annotate',
            'image': '$inputs.image',
            'predictions': '$steps.model.predictions',
        },
        {
            'type': 'roboflow_core/rosbridge_publish_sink@v1',
            'name': 'publish_detections',
            'host': ROSBRIDGE_HOST,
            'port': ROSBRIDGE_PORT,
            'topic': DETECTIONS_TOPIC,
            'message_type': 'vision_msgs/Detection2DArray',
            'payload': '$steps.model.predictions',
            'frame_id': 'camera',
        },
        {
            'type': 'roboflow_core/rosbridge_publish_sink@v1',
            'name': 'publish_annotated',
            'host': ROSBRIDGE_HOST,
            'port': ROSBRIDGE_PORT,
            'topic': ANNOTATED_TOPIC,
            'message_type': 'sensor_msgs/CompressedImage',
            'payload': '$steps.annotate.image',
            'frame_id': 'camera',
        },
    ],
    'outputs': [
        {'type': 'JsonField', 'name': 'predictions',
         'selector': '$steps.model.predictions'},
    ],
}

Submit the Workflow + a `rosbridge://` video reference to the inference
server. The `compression=cbor-raw` query param tells rosbridge to ship
raw bytes back instead of base64, skipping a decode round-trip on the
image hot path.

In [ ]:
import requests, urllib.parse

video_reference = (
    f'rosbridge://{ROSBRIDGE_HOST}:{ROSBRIDGE_PORT}{CAMERA_TOPIC}'
    f'?type={urllib.parse.quote("sensor_msgs/CompressedImage", safe="")}'
    f'&compression=cbor-raw'
)
print('video_reference =', video_reference)

payload = {
    'video_configuration': {
        'type': 'VideoConfiguration',
        'video_reference': video_reference,
    },
    'processing_configuration': {
        'type': 'WorkflowConfiguration',
        'workflow_specification': WORKFLOW,
        'image_input_name': 'image',
    },
    'sink_configuration': {'type': 'MemorySinkConfiguration'},
    'api_key': API_KEY,
}

resp = requests.post(f'{API_URL}/inference_pipelines/initialise', json=payload, timeout=30)
resp.raise_for_status()
pipeline_id = resp.json()['context']['pipeline_id']
print('Pipeline initialised:', pipeline_id)

## 5. Launch RViz 2

We run RViz inside `ros:humble` with X11 forwarded to the host display.
This works on a Linux desktop session; if you're on a headless box, skip
this cell and either point a remote RViz at the rosbridge endpoint,
or use [Foxglove Studio](https://app.foxglove.dev) which connects to
`ws://<host>:9090` directly — no install needed.

The bundled `inference.rviz` config sets up two `Image` displays:
the raw camera topic and the annotated overlay. The
`vision_msgs/Detection2DArray` topic is also published — to render
bounding boxes natively, install `vision_msgs_rviz_plugins` and add a
`Detection2DArray` display, or just rely on the annotated overlay.

In [ ]:
import os, subprocess

subprocess.run(['xhost', '+local:root'], check=False)  # no-op on Wayland

subprocess.run(['docker', 'rm', '-f', 'rviz'],
               check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

config_path = os.path.abspath('inference.rviz')
subprocess.run([
    'docker', 'run', '-d', '--name', 'rviz', '--network', 'host',
    '-e', f'DISPLAY={os.environ.get("DISPLAY", ":0")}',
    '-v', '/tmp/.X11-unix:/tmp/.X11-unix',
    '-v', f'{config_path}:/inference.rviz:ro',
    'osrf/ros:humble-desktop',
    'bash', '-c',
    'source /opt/ros/humble/setup.bash && rviz2 -d /inference.rviz'
], check=True)

print('RViz started — check `docker logs rviz` if no window appears.')

## 6. Replay sample video frames

We download a small demo clip and republish it as a stream of
`sensor_msgs/CompressedImage` messages. This stands in for `ros2 bag
play`; if you have a real bag with image topics, you can swap this cell
for `docker exec rosbridge ros2 bag play /path/to/bag` and skip the
Python publisher.

In [ ]:
import os, urllib.request

VIDEO_URL = ('https://media.roboflow.com/inference/notebooks/community/'
             'fitness_gpt_coach/squat_video.mp4')
VIDEO_PATH = 'sample.mp4'

if not os.path.exists(VIDEO_PATH):
    print('Downloading sample video …')
    urllib.request.urlretrieve(VIDEO_URL, VIDEO_PATH)
print('Video size:', os.path.getsize(VIDEO_PATH), 'bytes')

In [ ]:
import base64, threading, time
import cv2, roslibpy

PUBLISH_FPS = 10

image_topic = roslibpy.Topic(
    ros, CAMERA_TOPIC, 'sensor_msgs/CompressedImage', queue_size=1,
)
image_topic.advertise()

stop_event = threading.Event()

def publisher_loop():
    cap = cv2.VideoCapture(VIDEO_PATH)
    seq = 0
    period = 1.0 / PUBLISH_FPS
    try:
        while not stop_event.is_set():
            ok, frame = cap.read()
            if not ok:  # loop the clip
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                continue
            ok, jpeg = cv2.imencode('.jpg', frame, [int(cv2.IMWRITE_JPEG_QUALITY), 85])
            if not ok:
                continue
            now = time.time()
            secs = int(now)
            nsecs = int((now - secs) * 1e9)
            image_topic.publish(roslibpy.Message({
                'header': {
                    'stamp': {'sec': secs, 'nanosec': nsecs},
                    'frame_id': 'camera',
                },
                'format': 'jpeg',
                'data': base64.b64encode(jpeg.tobytes()).decode('ascii'),
            }))
            seq += 1
            time.sleep(period)
    finally:
        cap.release()
    print(f'publisher exited after {seq} frames')

thread = threading.Thread(target=publisher_loop, daemon=True)
thread.start()
print('Publishing to', CAMERA_TOPIC, 'at', PUBLISH_FPS, 'fps. RViz should now show frames.')

## 7. Confirm detections are flowing

Two ways to verify the pipeline is alive:

- Look at RViz — the `Annotated overlay` panel should update with
  bounding boxes drawn on the squat video.
- Subscribe to `/inference/detections` from this notebook and print
  the first message that arrives.

In [ ]:
import threading

got = threading.Event()
received = {}

def on_msg(msg):
    if got.is_set():
        return
    received.update(msg)
    got.set()

det_topic = roslibpy.Topic(
    ros, DETECTIONS_TOPIC, 'vision_msgs/Detection2DArray',
    queue_size=1, throttle_rate=200,
)
det_topic.subscribe(on_msg)

if got.wait(timeout=20):
    n = len(received.get('detections', []))
    print(f'received Detection2DArray with {n} detections')
    if n:
        first = received['detections'][0]
        print(' first bbox:', first.get('bbox'))
        print(' first class:', first.get('results', [{}])[0].get('hypothesis', {}))
else:
    print('no detections within 20s — check `docker logs rosbridge` and the inference server logs')

det_topic.unsubscribe()

## 8. Cleanup

Stops the publisher loop, terminates the server-side pipeline, and
removes the rosbridge / RViz containers.

In [ ]:
import subprocess

stop_event.set()
thread.join(timeout=5)

image_topic.unadvertise()
ros.terminate()

try:
    requests.post(
        f'{API_URL}/inference_pipelines/{pipeline_id}/terminate',
        timeout=10,
    ).raise_for_status()
    print('Pipeline terminated')
except Exception as exc:
    print('Pipeline terminate failed:', exc)

for name in ('rviz', 'rosbridge'):
    subprocess.run(['docker', 'rm', '-f', name], check=False,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('Done.')